In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
import torch
from parareal_grundfunktionen import load_OCP,

# This notebook was used to calculate the values in the tables in section 5, containing the suprema for all the different
# combination of FP´s, CP´s and OCP´s. It evaluates the convergence factor on a logarithmic grid over [1e-6,1e6]. Then the
# maxima and largest grid values are used as candidates. For every candidate, the gridpoint to the left and right is taken
# as an interval, on which then scipy.optimize.minmize_scalar is applied to the negative convergence factor, in order
# to approximate the supremum.

def approximate_supremum(function, lower, upper, gridpoints, top_k):

    x_grid = np.linspace(lower, upper, gridpoints)
    s_grid = 10.0**x_grid

    values = np.array(
        [float(function(float(s))) for s in s_grid],
        dtype=np.float64,
    )

    if not np.all(np.isfinite(values)):
        bad = np.flatnonzero(~np.isfinite(values))
        raise ValueError(
            f"Non-finite values encountered at {len(bad)} grid points."
        )

    local_indices = (
        np.flatnonzero(
            (values[1:-1] >= values[:-2])
            & (values[1:-1] >= values[2:])
        )
        + 1
    )

    k = min(top_k, gridpoints)
    top_indices = np.argpartition(values, -k)[-k:]

    candidate_indices = np.unique(
        np.concatenate((local_indices, top_indices))
    )

    grid_best_index = int(np.argmax(values))
    best_value = float(values[grid_best_index])
    best_s = float(s_grid[grid_best_index])

    for i in candidate_indices:
        if i == 0 or i == gridpoints - 1:
            continue

        x_left = x_grid[i - 1]
        x_right = x_grid[i + 1]

        result = minimize_scalar(
            lambda x: -float(function(10.0**x)),
            bounds=(x_left, x_right),
            method="bounded",
            options={
                "xatol": 1e-12,
                "maxiter": 500,
            },
        )

        candidate_value = -float(result.fun)
        candidate_s = 10.0 ** float(result.x)

        if candidate_value > best_value:
            best_value = candidate_value
            best_s = candidate_s

    diagnostics = {
        "left_boundary_value": float(values[0]),
        "right_boundary_value": float(values[-1]),
        "grid_best_s": float(s_grid[grid_best_index]),
        "grid_best_value": float(values[grid_best_index]),
        "grid_best_at_boundary": grid_best_index in (0, gridpoints - 1),
        "number_of_local_candidates": len(local_indices),
    }

    return best_value, best_s, diagnostics


# two-stage Lobatto IIIC
def r2(x):
    return (2) / (x ** 2 + 2 * x + 2)

# three-stage Lobatto IIIC
def r3(x):
    return (-6 * x + 24) / (x ** 3 + 6 * x ** 2 + 18 * x + 24)

# four-stage Lobatto IIIC
def r4(x):
    return (12*x**2 - 120*x + 360) / (x**4 + 12*x**3 + 72*x**2 + 240*x + 360)

# three-stage Radau IIA
def rc(s):
    b = 0.5 * (1 + np.sqrt(3) / 3)
    return 1 - s / (1 + b * s) - np.sqrt(3) / 6 * (s / (1 + b * s))**2

J=20


########LOBATTO IIIC 4-stage##########
Result_n1m2_LOBATTOIIIC4 = load_OCP("n1m2_001_100_LOBATTOIIIC4.pt")
R_n1m2_LOBATTOIIIC4 = Result_n1m2_LOBATTOIIIC4["R_scalar"]

Result_n1m3_LOBATTOIIIC4 = load_OCP("n1m3_001_100_LOBATTOIIIC4.pt")
R_n1m3_LOBATTOIIIC4 = Result_n1m3_LOBATTOIIIC4["R_scalar"]

Result_n2m3_LOBATTOIIIC4 = load_OCP("n2m3_001_100_LOBATTOIIIC4.pt")
R_n2m3_LOBATTOIIIC4 = Result_n2m3_LOBATTOIIIC4["R_scalar"]

Result_n2m4_LOBATTOIIIC4 = load_OCP("n2m4_001_100_LOBATTOIIIC4.pt")
R_n2m4_LOBATTOIIIC4 = Result_n2m4_LOBATTOIIIC4["R_scalar"]


def n1m2_LOBATTOIIIC4(s):
  return np.abs(r4(s/J)**J-R_n1m2_LOBATTOIIIC4(s))/(1 - np.abs(R_n1m2_LOBATTOIIIC4(s)))

def n1m3_LOBATTOIIIC4(s):
  return np.abs(r4(s/J)**J-R_n1m3_LOBATTOIIIC4(s))/(1 - np.abs(R_n1m3_LOBATTOIIIC4(s)))

def n2m3_LOBATTOIIIC4(s):
  return np.abs(r4(s/J)**J-R_n2m3_LOBATTOIIIC4(s))/(1 - np.abs(R_n2m3_LOBATTOIIIC4(s)))

def n2m4_LOBATTOIIIC4(s):
  return np.abs(r4(s/J)**J-R_n2m4_LOBATTOIIIC4(s))/(1 - np.abs(R_n2m4_LOBATTOIIIC4(s)))

best_value1, best_s1, diagnostics1 = approximate_supremum(n1m2_LOBATTOIIIC4, -6, 6, 50000, 20)
best_value2, best_s2, diagnostics2 = approximate_supremum(n1m3_LOBATTOIIIC4, -6, 6, 50000, 20)
best_value3, best_s3, diagnostics3 = approximate_supremum(n2m3_LOBATTOIIIC4, -6, 6, 50000, 20)
best_value4, best_s4, diagnostics4 = approximate_supremum(n2m4_LOBATTOIIIC4, -6, 6, 50000, 20)

print("LOBATTOIIIC4, n1m2", best_value1, best_s1)
print("LOBATTOIIIC4, n1m3", best_value2, best_s2)
print("LOBATTOIIIC4, n2m3", best_value3, best_s3)
print("LOBATTOIIIC4, n2m4", best_value4, best_s4)

print(round(best_value1,4),"&",round(best_value2,4),"&",round(best_value3,4),"&",round(best_value4,4),"\\")
print(round(best_s1,4),"&",round(best_s2,4),"&",round(best_s3,4),"&",round(best_s4,4),"\\")

# The suprema that are calculated here are only for the 4-stage LOBATTO IIIC scheme as a FP, for the rest of the values
# from the table in section 5, one can just copy this entry for another FP.
